# The PreDoc logit experiment, in `TrustRegionRadius.jl`

The PreDoc trains a discrete choice model by maximum likelihood on synthetic
data, inside a trust-region method whose sample size is chosen at every iteration
from a confidence statement about the achieved decrease. The sample-size rule is

$$N_{k+1} = \left\lceil \frac{\hat\sigma_k^2\,z_{1-\beta}^2}{\tilde\mu_k^2} \right\rceil,
\qquad \tilde\mu_k = \tilde f_k(x_k + s_k) - \tilde f_k(x_k),$$

with $\hat\sigma_k^2$ the variance of the individual decreases. The package
implements exactly this rule as `CertifiedDecrease`, on paired differences, with
$z_p$ in place of $z_{1-\beta}$ and the sign of $\tilde\mu_k$ flipped so that the
mean decrease is positive.

**What this notebook reproduces and what it cannot.** The reproduction is
faithful on the sample-size rule, the model Hessian, the subproblem solver, the
radius rule and the trust-region constants. It departs on three points, each
stated where it arises and collected in the last section.

Run it with

```
python -m nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=julia-1.11 notebooks/Sampling/predoc_logit_v1.ipynb
```

In [ ]:
const REPO = normpath(joinpath(@__DIR__, "..", ".."))
using Pkg; Pkg.activate(joinpath(REPO, "benchmark"))
include(joinpath(REPO, "benchmark", "initialisation.jl"))

const OUTDIR = joinpath(REPO, "notebooks", "Sampling")
mkpath(OUTDIR)

# The PreDoc settings, section "Logit experiment". Read off the source and not
# adjusted. eta_1 and eta_2 are the PreDoc's; in this package acceptance is
# decided by `eta` and scaling by `eta1` and `eta2`, and the PreDoc accepts on
# its eta_1, so eta = eta1 = 0.01 reproduces it.
const PD_ETA   = 0.01      # PreDoc eta_1, the acceptance threshold
const PD_ETA2  = 0.8       # PreDoc eta_2
const PD_G1    = 0.5       # PreDoc gamma_1
const PD_G2    = 0.9       # PreDoc gamma_2
const PD_G3    = 1.5       # PreDoc gamma_3
const PD_N0    = 100       # |S_0|
const PD_NMIN  = 10        # N_min
const PD_Z     = 1.6449    # z_{0.95}
const PD_P     = 10        # model dimension p
const PD_M     = 100_000   # population N
const PD_KMAX  = 300

println("PreDoc settings, transcribed from the source:")
@printf("  eta_1 = %.2f (acceptance), eta_2 = %.2f\n", PD_ETA, PD_ETA2)
@printf("  gamma = (%.2f, %.2f, %.2f)\n", PD_G1, PD_G2, PD_G3)
@printf("  |S_0| = %d, N_min = %d, z_{0.95} = %.4f\n", PD_N0, PD_NMIN, PD_Z)
@printf("  p = %d parameters, population N = %d\n", PD_P, PD_M)
println()
println("The package quantile is set from p, not passed directly. We solve for the p")
println("that reproduces z_{0.95}, and check it.")

## 1. The problem

The PreDoc uses a **multinomial** logit with five alternatives, $p = 10$
parameters and $N = 100\,000$ individuals, on synthetic data generated from the
model itself, so that the information identity holds and the BHHH approximation
is justified.

The package supplies `LogisticRegression`, which is the **binary** case of the
same family, also correctly specified by construction and also carrying the
identity exactly in the population. It is a `LikelihoodProblem`, which is what
makes `BHHHModel` legal over it.

**Departure 1.** Binary rather than five-alternative. The package has no
multinomial logit and we do not add one. The property the experiment turns on,
that the identity holds so BHHH is a genuine Hessian approximation near the
optimum, holds in both. The dimension and the population size are the PreDoc's.

`information_identity_error` measures the identity directly, so the premise is
checked rather than assumed.

In [ ]:
const PROB = LogisticRegression(K = PD_P, M = PD_M, seed = 0)
const BSTAR = β_true(PROB)

@printf("LogisticRegression: p = %d, M = %d\n", PROB.n, PROB.M)
@printf("population(prob)  = %d\n", population(PROB))
@printf("problem_class     = %s\n", string(problem_class(PROB)))
@printf("has_truth         = %s\n", has_truth(PROB))
@printf("required_problem(BHHHModel()) = %s\n", string(required_problem(BHHHModel())))
println()

# The identity at beta* and away from it. The PreDoc asserts the identity holds;
# here it is measured.
println("The information identity, measured rather than assumed")
println("`information_identity_error` returns four numbers, not one: B_err is the")
println("BHHH relative error, W_err the BHHH-2 one, and BW_gap the rank-one term")
println("||g g\'||/||H|| that separates them and vanishes at a stationary point.")
println("-"^92)
@printf("%-26s %14s %14s %14s %14s\n",
        "point", "B_err", "W_err", "BW_gap", "||g||")
println("-"^92)
IDENT = NamedTuple[]
for (lab, x) in (("beta*", BSTAR),
                 ("beta* + 0.5", BSTAR .+ 0.5),
                 ("zero (the PreDoc x_0)", zeros(PD_P)))
    e = information_identity_error(PROB, x)
    push!(IDENT, (label = lab, e...))
    @printf("%-26s %14.6f %14.6f %14.6f %14.4e\n",
            lab, e.B_err, e.W_err, e.BW_gap, e.grad_norm)
end
println()
println("Small at beta* and not small away from it, which is what makes BHHH a")
println("Hessian approximation near the optimum and a preconditioner far from it.")

## 2. The sample-size rule

`CertifiedDecrease` implements the PreDoc rule. Both use paired differences,
which is the point: holding the realisations fixed at both ends of the step makes
the individual decreases vanish pathwise as $s_k \to 0$, so the variance shrinks
with the step instead of staying $O(1)$.

The package parametrises the rule by a probability $p$ and derives $z_p$
internally, where the PreDoc passes $z_{1-\beta}$. We solve for the $p$ that
reproduces $z_{0.95} = 1.6449$ and check the result rather than assuming the
convention matches.

`monotone = true` gives $N_{k} \leq N_{k+1} \leq \texttt{growth}\cdot N_k$, which
is the PreDoc's $\text{Naive}(a, b)$ with $a = 1$. The PreDoc uses
$\text{Naive}(0.75, 2.0)$, which permits the sample to shrink.

**Departure 2.** A lower smoothing bound $a < 1$ is not available.
`monotone = false` leaves the size unbounded below and `monotone = true` forbids
any decrease. Both are run below and the difference is the subject of Figure 1.

In [ ]:
# Recover the package's p from the PreDoc's z, by bisection on the constructor.
function pd_p_for_z(ztarget)
    lo, hi = 0.5 + 1e-9, 1 - 1e-9
    for _ in 1:200
        mid = (lo + hi) / 2
        CertifiedDecrease(p = mid).z < ztarget ? (lo = mid) : (hi = mid)
    end
    return (lo + hi) / 2
end

const PD_PROB_LEVEL = pd_p_for_z(PD_Z)
@printf("z_{0.95} wanted        : %.6f\n", PD_Z)
@printf("p reproducing it       : %.6f\n", PD_PROB_LEVEL)
@printf("CertifiedDecrease(p).z : %.6f\n", CertifiedDecrease(p = PD_PROB_LEVEL).z)
@printf("difference             : %.3e\n",
        abs(CertifiedDecrease(p = PD_PROB_LEVEL).z - PD_Z))
println()
println("The package's `p` is a one-sided level and its `z` is taken at alpha = 2(1-p),")
println("so p = 0.95 would give a different constant. The value above is the one that")
println("matches the PreDoc's z, and it is checked and not assumed.")

## 3. The run

The radius rule is `RDelta`, which is the classical update of the PreDoc's Step 7:
$\gamma_3\Delta$ when $\rho \geq \eta_2$, $\gamma_2\Delta$ when
$\rho \in [\eta_1, \eta_2)$, $\gamma_1\Delta$ otherwise. The subproblem solver is
truncated conjugate gradient. The model is `BHHHModel`, the outer product of the
scores, which is the PreDoc's $\tilde B_k$.

The PreDoc sets $\Delta_0 = 0.1\|\tilde g_0\|$. The package takes a numeric
$\Delta_0$, so we evaluate $\|\tilde g_0\|$ on the initial batch and pass the
product.

In [ ]:
"""
    pd_oracle(rule; N_init, seed) -> FiniteSumNLP

The PreDoc oracle: the logit population as a finite sum, sampled by `rule`,
started from `x_0 = 0`.
"""
pd_oracle(rule; N_init = PD_N0, seed = 0) =
    FiniteSumNLP(PROB, rule; x0 = zeros(PD_P), seed = seed, N_init = N_init)

"""
    pd_params(Δ0) -> TRParams

`SOLVER_PARAMS` is not used: these are the PreDoc's own constants. Acceptance is
on `eta`, which is set to the PreDoc's `eta_1`.
"""
pd_params(Δ0) = TRParams(η = PD_ETA, η1 = PD_ETA, η2 = PD_ETA2,
                         Δ0 = Δ0, Δmin = 0.0, Δmax = Inf,
                         tol = 1e-6, max_iterations = PD_KMAX, max_time = 900.0)

"The initial sampled gradient norm, for Delta_0 = 0.1 ||g_0||."
function pd_delta0(rule; N_init = PD_N0, seed = 0)
    m = pd_oracle(rule; N_init = N_init, seed = seed)
    return 0.1 * norm(grad(m, zeros(PD_P)))
end

"""
    pd_run(rule; model, seed, label) -> NamedTuple

One `tr_solve` on the sampled logit, with every trajectory the figures read.
The rule, the model and the subsolver all come from the package.
"""
function pd_run(rule; model = BHHHModel(), seed = 0, label = "")
    m  = pd_oracle(rule; seed = seed)
    Δ0 = pd_delta0(rule; seed = seed)
    xs = [zeros(PD_P)]
    t0 = time()
    st = tr_solve(m; rule = RDelta(γ1 = PD_G1, γ2 = PD_G2, γ3 = PD_G3, Δmin = 0.0),
                  model = model, subsolver = SteihaugCG(),
                  params = pd_params(Δ0), trace = true,
                  callback = (_n, _s, s) -> push!(xs, copy(s.solution)))
    wall = time() - t0
    ss = st.solver_specific
    dist = [norm(p .- BSTAR) for p in xs]
    return (label = label, st = st, xs = xs, Δ0 = Δ0, wall = wall,
            N  = get(ss, :grad_sample_trajectory, Int[]),
            Δ  = get(ss, :delta_trajectory, Float64[]),
            g  = get(ss, :grad_trajectory, Float64[]),
            f  = get(ss, :obj_trajectory, Float64[]),
            ρ  = get(ss, :ratio_trajectory, Float64[]),
            s  = get(ss, :step_trajectory, Float64[]),
            cosc = get(ss, :cos_cauchy_trajectory, Float64[]),
            δpair = get(ss, :paired_decrease_trajectory, Float64[]),
            σpair = get(ss, :paired_variance_trajectory, Float64[]),
            tg = get(ss, :true_grad_trajectory, Float64[]),
            dist = dist, status = st.status, iter = st.iter,
            used = samples_used(m))
end

println("one run, to check the machinery before the sweep")
let r = pd_run(CertifiedDecrease(p = PD_PROB_LEVEL, N_min = PD_NMIN,
                                 N_start = PD_N0, monotone = false);
               label = "smoke")
    @printf("status %s in %d iters, Delta_0 = %.4e, wall %.1f s\n",
            string(r.status), r.iter, r.Δ0, r.wall)
    @printf("sample sizes: %d recorded, from %d to %d\n",
            length(r.N), isempty(r.N) ? 0 : minimum(r.N), isempty(r.N) ? 0 : maximum(r.N))
    @printf("paired stats recorded: %d decreases, %d variances\n",
            length(r.δpair), length(r.σpair))
    @printf("terms evaluated: grad %d, obj %d, total %d\n",
            r.used.grad, r.used.obj, r.used.total)
end

## 4. Figure 1: the sample size, unsmoothed and smoothed

The PreDoc reports that the raw rule is unstable from one iteration to the next
and introduces smoothing to control it. The two arms available here are
`monotone = false`, which is the raw rule, and `monotone = true`, which is
$\text{Naive}(1, \texttt{growth})$.

We measure the instability rather than describing it. The statistic is the median
of $|\log_2(N_{k+1}/N_k)|$, the typical size change per iteration in octaves.

In [ ]:
const PD_ARMS = [
    ("raw rule",                CertifiedDecrease(p = PD_PROB_LEVEL, N_min = PD_NMIN,
                                                  N_start = PD_N0, monotone = false)),
    ("Naive(1, 2), monotone",   CertifiedDecrease(p = PD_PROB_LEVEL, N_min = PD_NMIN,
                                                  N_start = PD_N0, monotone = true,
                                                  growth = 2.0)),
    ("Naive(1, 4), monotone",   CertifiedDecrease(p = PD_PROB_LEVEL, N_min = PD_NMIN,
                                                  N_start = PD_N0, monotone = true,
                                                  growth = 4.0)),
]

RUNS = Dict{String, Any}()
for (lab, rule) in PD_ARMS
    RUNS[lab] = pd_run(rule; label = lab)
end

pd_med(v) = isempty(v) ? NaN : (t = sort(v); m = length(t);
                                isodd(m) ? t[(m+1)÷2] : (t[m÷2] + t[m÷2+1]) / 2)

println("="^108)
println("TABLE 1. The sample size under the raw rule and under smoothing.")
println("`jump` is the median |log2(N_{k+1}/N_k)|, the typical change per iteration in octaves.")
println("="^108)
@printf("%-24s %10s %8s %8s %8s %10s %10s %12s %12s\n",
        "arm", "status", "iters", "N min", "N max", "N final", "jump", "grad terms", "total terms")
println("-"^108)
for (lab, _) in PD_ARMS
    r = RUNS[lab]
    jumps = length(r.N) < 2 ? Float64[] :
            [abs(log2(r.N[i+1] / r.N[i])) for i in 1:length(r.N)-1]
    @printf("%-24s %10s %8d %8d %8d %10d %10.4f %12d %12d\n",
            lab, string(r.status), r.iter,
            isempty(r.N) ? 0 : minimum(r.N), isempty(r.N) ? 0 : maximum(r.N),
            isempty(r.N) ? 0 : r.N[end], pd_med(jumps),
            r.used.grad, r.used.total)
end

In [ ]:
let plt = plot(xlabel = "iteration k", ylabel = "sample size N_k", yscale = :log10,
               legend = :bottomright, size = (760, 470),
               title = "Sample size, raw rule against Naive(1,b) smoothing",
               titlefontsize = 10)
    for (j, (lab, _)) in enumerate(PD_ARMS)
        r = RUNS[lab]
        isempty(r.N) && continue
        plot!(plt, 0:length(r.N)-1, r.N; label = lab, lw = 1.8,
              linestyle = (:solid, :dash, :dashdot)[mod1(j, 3)])
    end
    hline!(plt, [PD_M]; label = "population N = $PD_M", lw = 1.4,
           linestyle = :dot, color = :black)
    savefig(plt, joinpath(OUTDIR, "predoc_logit_fig1_sample_size.pdf"))
    display(plt)
end

## 5. Figure 2: the sample size against the distance to the solution

The PreDoc's central claim about the rule is that it uses small samples early and
raises them near a solution, because $\tilde\mu_k \to 0$ there while the variance
does not. The synthetic data has known generating parameters, so the distance
$\|x_k - \beta^\ast\|$ is available and the claim can be plotted directly rather
than inferred from the iteration index.

In [ ]:
let plt = plot(xlabel = "iteration k", ylabel = "value", yscale = :log10,
               legend = :left, size = (760, 470),
               title = "Sample size and distance to beta*, Naive(1,2)",
               titlefontsize = 10)
    r = RUNS["Naive(1, 2), monotone"]
    n = min(length(r.N), length(r.dist))
    plot!(plt, 0:n-1, max.(r.N[1:n], 1); label = "N_k", lw = 1.8)
    plot!(plt, 0:n-1, max.(r.dist[1:n], 1e-12); label = "||x_k - beta*||",
          lw = 1.6, linestyle = :dash)
    nd = min(length(r.Δ), n)
    plot!(plt, 0:nd-1, max.(r.Δ[1:nd], 1e-12); label = "Delta_k",
          lw = 1.4, linestyle = :dashdot)
    savefig(plt, joinpath(OUTDIR, "predoc_logit_fig2_size_vs_distance.pdf"))
    display(plt)
end

println("\n", "="^92)
println("Does the sample size rise as the iterate approaches beta*?")
println("Spearman-style check: the sign of the correlation between N_k and -dist_k.")
println("="^92)
@printf("%-24s %8s %14s %14s %14s\n", "arm", "n", "corr(N, -dist)", "N at k=0", "N at end")
println("-"^92)
for (lab, _) in PD_ARMS
    r = RUNS[lab]
    n = min(length(r.N), length(r.dist))
    n < 5 && continue
    a = Float64.(r.N[1:n]); b = -r.dist[1:n]
    ra = sortperm(sortperm(a)); rb = sortperm(sortperm(b))
    ā, b̄ = sum(ra)/n, sum(rb)/n
    num = sum((ra .- ā) .* (rb .- b̄))
    den = sqrt(sum(abs2, ra .- ā) * sum(abs2, rb .- b̄))
    @printf("%-24s %8d %14.4f %14d %14d\n", lab, n,
            den == 0 ? NaN : num / den, r.N[1], r.N[n])
end
println()
println("A positive correlation is the PreDoc's claim: the sample grows as the")
println("distance falls. It is a rank correlation and is reported as such.")

## 6. The two variance estimates, and why the comparison cannot be made here

The PreDoc compares the empirical variance of the individual decreases with the
outer-product approximation obtained from a first-order Taylor expansion,

$$\hat\sigma_k^2 \approx s_k^{\!\top}\tilde B_k s_k - \bigl(\tilde g_k^{\!\top}s_k\bigr)^2 .$$

**Departure 3.** The package feeds the sampling rule the empirical estimate only.
`paired_decrease_stats` computes the sample variance of the paired differences
and `record_paired!` hands that to the rule. The outer-product form is not wired
in and we do not wire it in.

We first tried to recover it after the fact from the trace, through
$\mathrm{pred}_k = \mathrm{ared}_k/\rho_k$ and
$\tilde g_k^{\!\top}s_k = -\cos_k\|\tilde g_k\|\|s_k\|$, which would give
$s_k^{\!\top}\tilde B_k s_k = -2(\mathrm{pred}_k + \tilde g_k^{\!\top}s_k)$.
**The reconstruction is wrong and the cell below is what shows it.** `BHHHModel`
is positive semidefinite by construction, so $s^{\!\top}Bs \geq 0$ at every
iteration, and the reconstruction produces negative values on a large fraction of
them.

Two independent reasons, either one fatal.

1. $\mathrm{ared}_k$ is not $f_k - f_{k+1}$ read off the objective trajectory. On
   a **rejected** step the iterate does not move, so the trajectory shows no
   change while the achieved reduction that entered $\rho_k$ was not zero.
2. Under sampling the batch changes at every iteration, so $f_k$ and $f_{k+1}$ are
   evaluated on **different batches**. Their difference mixes the step with the
   change of estimator, which is exactly the hazard `SequentialEstimation`
   documents when it explains why it is monotone by default.

So the comparison of the two variance estimates cannot be made from an archived
run at all. It needs the solver to record the predicted reduction, which the
trace does not carry: of its eighteen trajectories none is $\mathrm{pred}_k$ or
$s_k^{\!\top}B_ks_k$.

We report the failed check rather than a comparison built on it.

In [ ]:
"""
    pd_op_attempt(r) -> NamedTuple

The attempted reconstruction, kept so that the refutation is reproducible. The
`negatives` count is the refutation: an outer-product model cannot give
`s' B s < 0`.
"""
function pd_op_attempt(r)
    n = min(length(r.ρ), length(r.s), length(r.cosc), length(r.σpair),
            length(r.g) - 1, length(r.f) - 1)
    sBs = Float64[]; ks = Int[]
    for i in 1:n
        (isfinite(r.ρ[i]) && r.ρ[i] != 0 && isfinite(r.cosc[i])) || continue
        ared = r.f[i] - r.f[i+1]
        pred = ared / r.ρ[i]
        gTs  = -r.cosc[i] * r.g[i] * r.s[i]
        q    = -2 * (pred + gTs)
        isfinite(q) || continue
        push!(sBs, q); push!(ks, i - 1)
    end
    return (sBs = sBs, k = ks, neg = count(<(-1e-12), sBs))
end

println("="^100)
println("The reconstruction of s' B s from the trace, and its refutation.")
println("BHHHModel is positive semidefinite, so s' B s >= 0 at every iteration.")
println("A single negative value refutes the reconstruction.")
println("="^100)
@printf("%-24s %8s %16s %16s %12s %10s\n",
        "arm", "n", "min s'Bs", "median s'Bs", "negatives", "verdict")
println("-"^100)
OPCHECK = NamedTuple[]
for (lab, _) in PD_ARMS
    v = pd_op_attempt(RUNS[lab])
    isempty(v.sBs) && continue
    push!(OPCHECK, (label = lab, n = length(v.sBs), neg = v.neg,
                    mn = minimum(v.sBs)))
    @printf("%-24s %8d %16.6e %16.6e %12d %10s\n",
            lab, length(v.sBs), minimum(v.sBs), pd_med(v.sBs), v.neg,
            v.neg == 0 ? "ok" : "REFUTED")
end
println()
if any(c.neg > 0 for c in OPCHECK)
    println("The reconstruction is refuted. We do not report a variance comparison")
    println("built on it, and we do not adjust anything to make it pass.")
    println()
    println("What the package would need: one more trajectory in the trace, either")
    println("the predicted reduction pred_k or the curvature term s_k' B_k s_k. Both")
    println("are already formed inside `_tr_step!` to compute rho_k, so recording one")
    println("costs nothing beyond the storage.")
else
    println("The reconstruction survives on these runs. It would still be unsafe in")
    println("general, for the two reasons given above.")
end

In [ ]:
# The empirical estimate the rule actually used IS in the trace, and it is worth
# reporting on its own: it is what drove every sample size above.
println("\n", "="^96)
println("TABLE 2. The paired variance and the paired decrease, as the rule used them.")
println("These are recorded by the solver and are not reconstructed.")
println("="^96)
@printf("%-24s %8s %14s %14s %14s %14s\n",
        "arm", "n", "median sigma^2", "median delta", "min delta", "max sigma^2")
println("-"^96)
for (lab, _) in PD_ARMS
    r = RUNS[lab]
    good = [i for i in eachindex(r.σpair)
            if isfinite(r.σpair[i]) && isfinite(r.δpair[i])]
    isempty(good) && continue
    @printf("%-24s %8d %14.4e %14.4e %14.4e %14.4e\n",
            lab, length(good), pd_med(r.σpair[good]), pd_med(r.δpair[good]),
            minimum(r.δpair[good]), maximum(r.σpair[good]))
end
println()
println("A negative paired decrease is a batch that failed to certify any decrease.")
println("CertifiedDecrease grows the sample by `growth` there, on the reading that a")
println("failure to certify is evidence the batch was too small.")

In [ ]:
let plt = plot(xlabel = "iteration k", ylabel = "value", yscale = :log10,
               legend = :topright, size = (760, 470),
               title = "The paired statistics the rule used, Naive(1,2)",
               titlefontsize = 10)
    r = RUNS["Naive(1, 2), monotone"]
    good = [i for i in eachindex(r.σpair)
            if isfinite(r.σpair[i]) && r.σpair[i] > 0]
    gd = [i for i in eachindex(r.δpair) if isfinite(r.δpair[i]) && r.δpair[i] > 0]
    isempty(good) || plot!(plt, good .- 1, r.σpair[good];
                           label = "paired variance", lw = 1.7)
    isempty(gd) || plot!(plt, gd .- 1, r.δpair[gd];
                         label = "paired decrease (positive part)", lw = 1.5,
                         linestyle = :dash)
    savefig(plt, joinpath(OUTDIR, "predoc_logit_fig3_paired.pdf"))
    display(plt)
end

## 7. Figure 4: the radius under two model Hessians

The PreDoc plots $\Delta_k$ under a subsampled true Hessian and under BHHH, and
reads the second as evidence that the outer product becomes a good approximation
near the solution: the radius stops contracting and begins to grow.

Here the two models are `ExactHessian`, which on a `FiniteSumNLP` is the Hessian
of the sampled objective, and `BHHHModel`. The sampling rule is the same in both.

In [ ]:
const PD_MODELS = [("BHHH", () -> BHHHModel()),
                   ("ExactHessian", () -> ExactHessian())]

MODEL_RUNS = Dict{String, Any}()
for (mnm, mk) in PD_MODELS
    MODEL_RUNS[mnm] = pd_run(CertifiedDecrease(p = PD_PROB_LEVEL, N_min = PD_NMIN,
                                               N_start = PD_N0, monotone = true,
                                               growth = 2.0);
                             model = mk(), label = mnm)
end

println("="^104)
println("TABLE 3. The same sampling rule under two model Hessians.")
println("="^104)
@printf("%-16s %12s %8s %12s %12s %12s %12s %12s\n",
        "model", "status", "iters", "Delta_0", "Delta_end", "dist end", "|g| end", "total terms")
println("-"^104)
for (mnm, _) in PD_MODELS
    r = MODEL_RUNS[mnm]
    @printf("%-16s %12s %8d %12.4e %12.4e %12.4e %12.4e %12d\n",
            mnm, string(r.status), r.iter, r.Δ0,
            isempty(r.Δ) ? NaN : r.Δ[end], r.dist[end],
            Float64(r.st.dual_feas), r.used.total)
end

In [ ]:
let plt = plot(xlabel = "iteration k", ylabel = "Delta_k", yscale = :log10,
               legend = :bottomright, size = (760, 470),
               title = "Trust-region radius under two model Hessians",
               titlefontsize = 10)
    for (j, (mnm, _)) in enumerate(PD_MODELS)
        r = MODEL_RUNS[mnm]
        isempty(r.Δ) && continue
        plot!(plt, 0:length(r.Δ)-1, max.(r.Δ, 1e-300); label = mnm, lw = 1.8,
              linestyle = j == 1 ? :solid : :dash)
    end
    savefig(plt, joinpath(OUTDIR, "predoc_logit_fig4_radius.pdf"))
    display(plt)
end

## 8. What could not be reproduced

Three departures, each stated where it arose.

In [ ]:
println("="^100)
println("Departures from the PreDoc experiment, and what each would need.")
println("="^100)
for (n, what, why, need) in (
    (1, "binary rather than 5-alternative logit",
        "the package has no multinomial logit; LogisticRegression is the binary case",
        "a MultinomialLogit <: LikelihoodProblem with per-observation scores"),
    (2, "no lower smoothing bound, Naive(a, b) with a < 1",
        "CertifiedDecrease offers monotone=false (unbounded) or monotone=true (a = 1)",
        "a floor field on the rule, or a SmoothedSize wrapper over a SamplingRule"),
    (3, "the outer-product variance is not wired to the rule",
        "record_paired! receives the empirical estimate from paired_decrease_stats only",
        "a variance keyword on CertifiedDecrease selecting empirical or outer product"),
    (4, "only the VAI sample scheme",
        "_draw! redraws independently at every iteration; VAC and I/CRV are absent",
        "a scheme field on the oracle: :independent, :nested, :prefix"))
    @printf("\n%d. %s\n", n, what)
    @printf("   why  : %s\n", why)
    @printf("   needs: %s\n", need)
end

println("\n", "="^100)
println("What WAS reproduced without any addition to the package")
println("="^100)
for line in ("the sample-size rule itself, as CertifiedDecrease on paired differences",
             "the paired-difference variance, as paired_decrease_stats",
             "the BHHH model, as BHHHModel over a LikelihoodProblem",
             "the information identity, measured by information_identity_error",
             "the radius rule of Step 7, as RDelta at the PreDoc's gamma",
             "the acceptance test, as eta = eta_1",
             "truncated CG, as SteihaugCG",
             "Delta_0 = 0.1 ||g_0||, by evaluating the initial sampled gradient",
             "the cost measure, as samples_used rather than iteration counts")
    println("   ", line)
end

In [ ]:
println("\n", "="^80)
println("Closing summary")
println("="^80)
@printf("problem     : LogisticRegression, p = %d, M = %d, B_err at beta* = %.3e\n",
        PD_P, PD_M, information_identity_error(PROB, BSTAR).B_err)
@printf("rule        : CertifiedDecrease(p = %.6f) reproducing z = %.4f\n",
        PD_PROB_LEVEL, PD_Z)
for (lab, _) in PD_ARMS
    r = RUNS[lab]
    @printf("%-24s %10s in %3d iters, N from %6d to %6d, %d terms\n",
            lab, string(r.status), r.iter,
            isempty(r.N) ? 0 : minimum(r.N), isempty(r.N) ? 0 : maximum(r.N),
            r.used.total)
end
println()
println("Figures written to ", OUTDIR)
for f in ("predoc_logit_fig1_sample_size.pdf", "predoc_logit_fig2_size_vs_distance.pdf",
          "predoc_logit_fig3_paired.pdf", "predoc_logit_fig4_radius.pdf")
    @printf("   %-44s %s\n", f, isfile(joinpath(OUTDIR, f)) ? "ok" : "MISSING")
end
println("\nDONE.")